In [ ]:

import pandas as pd
import numpy as np


file_path = r"C:\Users\Admin\Downloads\agri_dataset.csv"
df = pd.read_csv(file_path)



NUMBER 2


In [ ]:
print(df.head(10))


   record_id observation_date         region    crop  rainfall_mm  \
0          1       1990-01-01  Southern Asia  Barley       165.00   
1          2       1991-01-01  Southern Asia  Barley       218.00   
2          3       1992-01-01  Southern Asia  Barley       241.31   
3          4       1993-01-01  Southern Asia  Barley       187.41   
4          5       1994-01-01  Southern Asia  Barley       194.99   
5          6       1995-01-01  Southern Asia  Barley       144.28   
6          7       1996-01-01  Southern Asia  Barley       198.33   
7          8       1997-01-01  Southern Asia  Barley       220.16   
8          9       1998-01-01  Southern Asia  Barley       271.10   
9         10       1999-01-01  Southern Asia  Barley       122.19   

   temperature_c  humidity_pct  soil_moisture     yield_kg  plot_area_ha  \
0          7.280        41.915          0.487  216000000.0      205000.0   
1          6.275        47.304          0.498  217000000.0      204000.0   
2          5

In [5]:
# View Last 5 Rows
print(df.tail(5))

       record_id observation_date          region   crop  rainfall_mm  \
36659      36660       2018-01-01  Eastern Africa  Wheat       699.02   
36660      36661       2019-01-01  Eastern Africa  Wheat       449.31   
36661      36662       2020-01-01  Eastern Africa  Wheat       777.30   
36662      36663       2021-01-01  Eastern Africa  Wheat       889.88   
36663      36664       2022-01-01  Eastern Africa  Wheat       783.96   

       temperature_c  humidity_pct  soil_moisture     yield_kg  plot_area_ha  \
36659         20.395        57.478          0.449   45000000.0       16558.0   
36660         21.515        51.248          0.414   94685000.0       24186.0   
36661         20.983        54.999          0.426  212530000.0       44466.0   
36662         19.388        63.689          0.495  337212000.0       66434.0   
36663         19.992        61.849          0.451  250000000.0       48586.0   

        country iso3  year  yield_kg_ha  
36659  Zimbabwe  ZWE  2018      2717.7

In [ ]:
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")

Dataset Shape: 36664 rows, 14 columns


In [ ]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 36664 entries, 0 to 36663
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   record_id         36664 non-null  int64  
 1   observation_date  36664 non-null  str    
 2   region            36664 non-null  str    
 3   crop              36664 non-null  str    
 4   rainfall_mm       36664 non-null  float64
 5   temperature_c     36664 non-null  float64
 6   humidity_pct      36664 non-null  float64
 7   soil_moisture     36664 non-null  float64
 8   yield_kg          36664 non-null  float64
 9   plot_area_ha      36664 non-null  float64
 10  country           36664 non-null  str    
 11  iso3              36664 non-null  str    
 12  year              36664 non-null  int64  
 13  yield_kg_ha       36664 non-null  float64
dtypes: float64(7), int64(2), str(5)
memory usage: 3.9 MB
None


In [ ]:
print(df.dtypes)


record_id             int64
observation_date        str
region                  str
crop                    str
rainfall_mm         float64
temperature_c       float64
humidity_pct        float64
soil_moisture       float64
yield_kg            float64
plot_area_ha        float64
country                 str
iso3                    str
year                  int64
yield_kg_ha         float64
dtype: object


In [ ]:
print(df.describe(include="all"))

           record_id observation_date          region   crop   rainfall_mm  \
count   36664.000000            36664           36664  36664  36664.000000   
unique           NaN               33              22     10           NaN   
top              NaN       2017-01-01  Eastern Africa  Maize           NaN   
freq             NaN             1140            4176   5167           NaN   
mean    18332.500000              NaN             NaN    NaN    954.706168   
std     10584.129471              NaN             NaN    NaN    718.162717   
min         1.000000              NaN             NaN    NaN      0.000000   
25%      9166.750000              NaN             NaN    NaN    426.430000   
50%     18332.500000              NaN             NaN    NaN    841.650000   
75%     27498.250000              NaN             NaN    NaN   1333.500000   
max     36664.000000              NaN             NaN    NaN   6989.040000   

        temperature_c  humidity_pct  soil_moisture      yield_k

Number 3

In [11]:
EXPECTED_COLUMNS = {
    "record_id": "int64", "observation_date": "datetime64", "region": "str", "crop": "str",
    "rainfall_mm": "float64", "temperature_c": "float64", "humidity_pct": "float64",
    "soil_moisture": "float64", "yield_kg": "float64", "plot_area_ha": "float64",
    "country": "str", "iso3": "str", "year": "int64", "yield_kg_ha": "float64",
}


def validate_columns(df, expected=EXPECTED_COLUMNS):
    """Compare df to the expected schema. Returns a table with one row per column and a status."""
    rows = []
    for col, exp in expected.items():
        if col not in df.columns:
            rows.append((col, exp, "-", "MISSING"))
            continue
        actual = str(df[col].dtype)
        ok = actual.startswith(exp) or (exp == "str" and actual in ("object", "str", "string"))
        rows.append((col, exp, actual, "OK" if ok else "TYPE MISMATCH"))
    for col in df.columns:
        if col not in expected:
            rows.append((col, "-", str(df[col].dtype), "UNEXPECTED"))
    return pd.DataFrame(rows, columns=["column", "expected", "actual", "status"])


def summarize_missingness(df):
    """Null count and percentage for every column, worst first."""
    out = pd.DataFrame({
        "null_count": df.isna().sum(),
        "null_pct": (df.isna().mean() * 100).round(2),
    })
    return out.sort_values("null_count", ascending=False)


def check_ranges(df, rules):
    """rules = {description: boolean Series that is True for BAD rows}. Returns violation counts."""
    return pd.DataFrame(
        {"rule": list(rules), "violations": [int(bad.sum()) for bad in rules.values()]}
    )


def flag_outliers_iqr(df, value_col, group_col):
    """Flag (do NOT delete) values outside 1.5 x IQR of their own group. Returns a boolean Series."""
    q1 = df.groupby(group_col)[value_col].transform(lambda s: s.quantile(0.25))
    q3 = df.groupby(group_col)[value_col].transform(lambda s: s.quantile(0.75))
    iqr = q3 - q1
    return (df[value_col] < q1 - 1.5 * iqr) | (df[value_col] > q3 + 1.5 * iqr)

In [12]:
validate_columns(df)

,column,expected,actual,status
0,record_id,int64,int64,OK
1,observation_date,datetime64,str,TYPE MISMATCH
2,region,str,str,OK
3,crop,str,str,OK
4,rainfall_mm,float64,float64,OK
5,temperature_c,float64,float64,OK
6,humidity_pct,float64,float64,OK
7,soil_moisture,float64,float64,OK
8,yield_kg,float64,float64,OK
9,plot_area_ha,float64,float64,OK
